In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

## BufferMemory

In [3]:
from langchain.memory import ConversationBufferMemory

In [4]:
memory = ConversationBufferMemory()
memory

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8300\2223904900.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]))

In [5]:
memory.save_context(
    inputs={
        "human": "Hello, I want to open a bank account remotely. How do I start?",
    },
    outputs={
        "ai": "Hello! I'm glad you want to open an account. First, please prepare your ID for identity verification."
    },
)

In [6]:
memory

ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, I want to open a bank account remotely. How do I start?', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! I'm glad you want to open an account. First, please prepare your ID for identity verification.", additional_kwargs={}, response_metadata={})]))

In [7]:
print(memory.load_memory_variables({})['history'])

Human: Hello, I want to open a bank account remotely. How do I start?
AI: Hello! I'm glad you want to open an account. First, please prepare your ID for identity verification.


In [18]:
from operator import itemgetter

from langchain_openai.chat_models import ChatOpenAI

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pydantic import BaseModel, Field
from langchain_core.runnables import (
    RunnableLambda,
    ConfigurableFieldSpec,
    RunnablePassthrough,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

In [13]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You're an assistant who's good at {ability}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

In [14]:
chain = prompt | llm

In [24]:
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """In memory implementation of chat message history."""

    messages: list[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add a list of messages to the store"""
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []

In [25]:
store = {}

In [26]:
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

In [27]:
history = get_by_session_id("1")
history.add_message(AIMessage(content="hello"))
print(store)  # noqa: T201

{'1': InMemoryHistory(messages=[AIMessage(content='hello', additional_kwargs={}, response_metadata={})])}


In [28]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='question',
    history_messages_key='history'
)

In [29]:
print(chain_with_history.invoke(  # noqa: T201
    {"ability": "math", "question": "What does cosine mean?"},
    config={"configurable": {"session_id": "foo"}}
))

content='Cosine is a trigonometric function that relates the angles and lengths of sides in a right triangle. Specifically, for an angle \\( \\theta \\) in a right triangle, the cosine of that angle is defined as the ratio of the length of the adjacent side (the side next to the angle) to the length of the hypotenuse (the longest side of the triangle):\n\n\\[\n\\cos(\\theta) = \\frac{\\text{Adjacent}}{\\text{Hypotenuse}}\n\\]\n\nCosine is also defined for all real numbers using the unit circle, where the cosine of an angle corresponds to the x-coordinate of the point on the unit circle that is at that angle from the positive x-axis. The cosine function is periodic with a period of \\( 2\\pi \\) radians or 360 degrees, and it ranges from -1 to 1.\n\nIn addition to its geometric interpretation, the cosine function has many applications in physics, engineering, and various fields of mathematics.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 2

In [30]:
print(store)

{'1': InMemoryHistory(messages=[AIMessage(content='hello', additional_kwargs={}, response_metadata={})]), 'foo': InMemoryHistory(messages=[HumanMessage(content='What does cosine mean?', additional_kwargs={}, response_metadata={}), AIMessage(content='Cosine is a trigonometric function that relates the angles and lengths of sides in a right triangle. Specifically, for an angle \\( \\theta \\) in a right triangle, the cosine of that angle is defined as the ratio of the length of the adjacent side (the side next to the angle) to the length of the hypotenuse (the longest side of the triangle):\n\n\\[\n\\cos(\\theta) = \\frac{\\text{Adjacent}}{\\text{Hypotenuse}}\n\\]\n\nCosine is also defined for all real numbers using the unit circle, where the cosine of an angle corresponds to the x-coordinate of the point on the unit circle that is at that angle from the positive x-axis. The cosine function is periodic with a period of \\( 2\\pi \\) radians or 360 degrees, and it ranges from -1 to 1.\n\n

In [31]:
print(chain_with_history.invoke(  # noqa: T201
    {"ability": "math", "question": "What's its inverse"},
    config={"configurable": {"session_id": "foo"}}
))

print(store) 

content='The inverse of the cosine function is called the arccosine function, denoted as \\( \\arccos(x) \\) or sometimes \\( \\cos^{-1}(x) \\). The arccosine function takes a value in the range of \\([-1, 1]\\) (which is the range of the cosine function) and returns an angle \\( \\theta \\) in the range \\([0, \\pi]\\) radians (or \\([0, 180^\\circ]\\)) such that:\n\n\\[\n\\cos(\\theta) = x\n\\]\n\nIn other words, if you know the cosine of an angle and want to find the angle itself, you can use the arccosine function. For example:\n\n\\[\n\\theta = \\arccos(0.5)\n\\]\n\nThis will give you \\( \\theta = \\frac{\\pi}{3} \\) radians (or \\( 60^\\circ \\)), since \\( \\cos\\left(\\frac{\\pi}{3}\\right) = 0.5 \\). \n\nThe arccosine function is useful in various mathematical applications, including solving triangles and analyzing periodic functions.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 238, 'prompt_tokens': 234, 'total_tokens': 472, 'c

In [35]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=5, return_messages=True)

In [37]:
memory.save_context(
    inputs={
        "human": "Hello, I'd like to open a bank account online. How do I get started?"
    },
    outputs={
        "ai": "Hello! I'm glad you want to open an account. First, could you please prepare your ID for verification?"
    },
)
memory.save_context(
    inputs={"human": "Yes, I have my ID ready. What should I do next?"},
    outputs={
        "ai": "Thank you. Please take clear photos of both sides of your ID and upload them. Then we'll proceed with the identity verification process."
    },
)
memory.save_context(
    inputs={
        "human": "I've uploaded the photos. How does the identity verification work?"
    },
    outputs={
        "ai": "I've confirmed your uploaded photos. Now, please proceed with phone verification. You'll receive a verification code via text message - please enter that code."
    },
)
memory.save_context(
    inputs={
        "human": "I've entered the verification code. How do I proceed with opening the account?"
    },
    outputs={
        "ai": "Your identity has been verified. Now, please select the type of account you want and enter the required information. You can choose the type of deposit, currency, and other options."
    },
)
memory.save_context(
    inputs={"human": "I've entered all the information. What's the next step?"},
    outputs={
        "ai": "I've confirmed your information. The account opening process is almost complete. Finally, please agree to the terms of service and confirm the account opening."
    },
)
memory.save_context(
    inputs={"human": "I've completed all the steps. Has the account been opened?"},
    outputs={
        "ai": "Yes, your account has been successfully opened. Your account number and related information have been sent to your registered email. Please let us know if you need any further assistance. Thank you!"
    },
)

In [38]:
# Check the conversation history
memory.load_memory_variables({})["history"]

[HumanMessage(content='Yes, I have my ID ready. What should I do next?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Thank you. Please take clear photos of both sides of your ID and upload them. Then we'll proceed with the identity verification process.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="I've uploaded the photos. How does the identity verification work?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="I've confirmed your uploaded photos. Now, please proceed with phone verification. You'll receive a verification code via text message - please enter that code.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="I've entered the verification code. How do I proceed with opening the account?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your identity has been verified. Now, please select the type of account you want and enter the required information. You can choose the type of d

## Entity Memory

In [40]:
from langchain.memory.entity import ConversationEntityMemory
from langchain.prompts import PromptTemplate

In [41]:
entity_memory_conversation_template = PromptTemplate(
    input_variables=["entities", "history", "input"],
    template="""
You are an assistant to a human, powered by a large language model trained by OpenAI.

You assist with various tasks, from answering simple questions to providing detailed discussions on a wide range of topics. You can generate human-like text, allowing natural conversations and coherent, relevant responses.

You constantly learn and improve, processing large amounts of text to provide accurate and informative responses. You can use personalized information provided in the context below, along with your own generated knowledge.

Context:
{entities}

Current conversation:
{history}
Last line:
Human: {input}
You:
""",
)

In [42]:
print(entity_memory_conversation_template)

input_variables=['entities', 'history', 'input'] input_types={} partial_variables={} template='\nYou are an assistant to a human, powered by a large language model trained by OpenAI.\n\nYou assist with various tasks, from answering simple questions to providing detailed discussions on a wide range of topics. You can generate human-like text, allowing natural conversations and coherent, relevant responses.\n\nYou constantly learn and improve, processing large amounts of text to provide accurate and informative responses. You can use personalized information provided in the context below, along with your own generated knowledge.\n\nContext:\n{entities}\n\nCurrent conversation:\n{history}\nLast line:\nHuman: {input}\nYou:\n'


In [43]:
chain = entity_memory_conversation_template | llm

In [44]:
conversation = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='input',
    history_messages_key='history'
)

In [45]:
response = conversation.invoke(
    {"input": "What's the capital of France?", "entities": "User likes geography."},
    config={"configurable": {"session_id": "user123"}}
)

In [46]:
print(response.content)

The capital of France is Paris. It's known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. Paris is also famous for its rich history, art, and culture. Would you like to know more about Paris or any other geographic topic?


In [50]:
print(store["user123"])

Human: What's the capital of France?
AI: The capital of France is Paris. It's known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. Paris is also famous for its rich history, art, and culture. Would you like to know more about Paris or any other geographic topic?


## KG memory

In [51]:
from langchain_community.memory.kg import ConversationKGMemory

In [52]:
memory = ConversationKGMemory(llm=llm, return_messages=True)
memory.save_context(
    {"input": "This is Shelly Kim who lives in Pangyo."},
    {"output": "Hello Shelly, nice to meet you! What kind of work do you do?"},
)
memory.save_context(
    {"input": "Shelly Kim is our company's new designer."},
    {
        "output": "That's great! Welcome to our team. I hope you'll enjoy working with us."
    },
)

In [53]:
memory.kg.get_topological_sort()

['Shelly Kim', 'Pangyo', "our company's new designer"]

In [54]:
memory.get_current_entities({"input": "Who is Shelly Kim?"})

['Shelly Kim']

In [58]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """The following is a friendly conversation between a human and an AI. 
The AI is talkative and provides lots of specific details from its context. 
If the AI does not know the answer to a question, it truthfully says it does not know. 
The AI ONLY uses information contained in the "Relevant Information" section and does not hallucinate.

Relevant Information:
{history}""",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

memory = ConversationKGMemory(llm=llm, return_messages=True, memory_key="history")

In [59]:
class ConversationChain:
    def __init__(self, prompt, llm, memory):
        self.memory = memory
        self.chain = (
            RunnablePassthrough()
            | RunnablePassthrough.assign(
                history=RunnableLambda(memory.load_memory_variables)
                | itemgetter("history")
            )
            | prompt
            | llm
        )
    
    def invoke(self, input_dict):
        response = self.chain.invoke(input_dict)
        self.memory.save_context(input_dict, {"output": response.content})
        return response


In [60]:
conversation_with_kg = ConversationChain(prompt, llm, memory)

In [61]:
response = conversation_with_kg.invoke(
    {
        "input": "My name is Teddy. Shelly is a coworker of mine, and she's a new designer at our company."
    }
)
response.content

"Nice to meet you, Teddy! It's great that you have a new coworker like Shelly on your team. How are things going with her as she settles into the role?"

In [62]:
conversation_with_kg.memory.load_memory_variables({"input": "who is Shelly?"})

{'history': [SystemMessage(content='On Shelly: Shelly is a coworker. Shelly is a new designer. Shelly is at our company.', additional_kwargs={}, response_metadata={})]}

## Summary Memory

In [1]:
from langchain.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14856\1239179074.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm, return_messages=True)


In [2]:
memory.save_context(
    inputs={"human": "What is the price of the Europe travel package?"},
    outputs={
        "ai": "The base price for the 14-night, 15-day Europe package is €3,500. This price includes airfare, hotel accommodations, and admission fees to designated tourist attractions. Additional costs may vary depending on optional tours or personal expenses."
    },
)
memory.save_context(
    inputs={"human": "What are the main tourist attractions included in the trip?"},
    outputs={
        "ai": "This trip includes visits to famous European landmarks such as the Eiffel Tower in Paris, the Colosseum in Rome, the Brandenburg Gate in Berlin, and Rhine Falls in Zurich. You'll comprehensively experience iconic sites in each city."
    },
)
memory.save_context(
    inputs={"human": "Is travel insurance included?"},
    outputs={
        "ai": "Yes, basic travel insurance is provided for all travelers. This insurance includes medical expense coverage and support in emergency situations. Enhanced coverage is available upon request."
    },
)
memory.save_context(
    inputs={
        "human": "Can I upgrade my flight seat to business class? How much does it cost?"
    },
    outputs={
        "ai": "Upgrading your flight seat to business class is possible. The upgrade cost is approximately €1,200 round-trip. Business class offers benefits such as wider seats, premium in-flight meals, and additional baggage allowance."
    },
)
memory.save_context(
    inputs={"human": "What is the hotel rating included in the package?"},
    outputs={
        "ai": "The package includes accommodation in 4-star hotels. Each hotel offers comfort and convenience, with central locations providing easy access to tourist sites. All hotels are equipped with excellent services and amenities."
    },
)
memory.save_context(
    inputs={"human": "Can you provide more details about the meal options?"},
    outputs={
        "ai": "This travel package includes daily breakfast served at the hotel. Lunch and dinner are not included, giving travelers the flexibility to explore and enjoy a variety of local cuisines. Additionally, a list of recommended restaurants in each city is provided to help maximize your culinary experience."
    },
)
memory.save_context(
    inputs={
        "human": "How much is the deposit for booking the package? What is the cancellation policy?"
    },
    outputs={
        "ai": "A deposit of €500 is required when booking the package. The cancellation policy allows a full refund if canceled at least 30 days before the booking date. After that, the deposit becomes non-refundable. If canceled within 14 days of the travel start date, 50% of the total cost will be charged, and after that, the full cost will be non-refundable."
    },
)

In [4]:
# Check stored memory.
print(memory.load_memory_variables({})["history"])

[SystemMessage(content='The human asks about the price of the Europe travel package, and the AI responds that the base price for a 14-night, 15-day package is €3,500, which includes airfare, accommodations, and admission fees to attractions, mentioning that additional costs may vary based on optional tours or personal expenses. The human inquires about the main tourist attractions included in the trip, and the AI explains that they include landmarks such as the Eiffel Tower, the Colosseum, the Brandenburg Gate, and Rhine Falls. The human then asks if travel insurance is included, and the AI confirms that basic travel insurance covering medical expenses and emergencies is provided, with enhanced coverage available upon request. The human further inquires about the cost of upgrading to business class, to which the AI informs them that the upgrade is possible for approximately €1,200 round-trip, highlighting benefits like wider seats, premium meals, and additional baggage allowance. The h

In [5]:
from langchain.memory import ConversationSummaryBufferMemory
memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=200,
    return_messages=True
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14856\4197024960.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(


In [6]:
memory.save_context(
    inputs={"human": "What is the price of the Europe travel package?"},
    outputs={
        "ai": "The base price for the 14-night, 15-day Europe package is €3,500. This price includes airfare, hotel accommodations, and admission fees to designated tourist attractions. Additional costs may vary depending on optional tours or personal expenses."
    },
)

In [7]:
# Check the stored conversation history in memory
memory.load_memory_variables({})["history"]

[HumanMessage(content='What is the price of the Europe travel package?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The base price for the 14-night, 15-day Europe package is €3,500. This price includes airfare, hotel accommodations, and admission fees to designated tourist attractions. Additional costs may vary depending on optional tours or personal expenses.', additional_kwargs={}, response_metadata={})]

In [8]:
memory.save_context(
    inputs={"human": "What are the main tourist attractions included in the trip?"},
    outputs={
        "ai": "This trip includes visits to famous European landmarks such as the Eiffel Tower in Paris, the Colosseum in Rome, the Brandenburg Gate in Berlin, and Rhine Falls in Zurich. You'll comprehensively experience iconic sites in each city."
    },
)
memory.save_context(
    inputs={"human": "Is travel insurance included?"},
    outputs={
        "ai": "Yes, basic travel insurance is provided for all travelers. This insurance includes medical expense coverage and support in emergency situations. Enhanced coverage is available upon request."
    },
)
memory.save_context(
    inputs={
        "human": "Can I upgrade my flight seat to business class? How much does it cost?"
    },
    outputs={
        "ai": "Upgrading your flight seat to business class is possible. The upgrade cost is approximately €1,200 round-trip. Business class offers benefits such as wider seats, premium in-flight meals, and additional baggage allowance."
    },
)
memory.save_context(
    inputs={"human": "What is the hotel rating included in the package?"},
    outputs={
        "ai": "The package includes accommodation in 4-star hotels. Each hotel offers comfort and convenience, with central locations providing easy access to tourist sites. All hotels are equipped with excellent services and amenities."
    },
)

In [9]:
memory.load_memory_variables({})["history"]

[SystemMessage(content='The human inquires about the price of the Europe travel package, and the AI responds that the base price for the 14-night, 15-day package is €3,500, which includes airfare, hotel accommodations, and admission fees, with additional costs for optional tours or personal expenses. The human then asks about the main tourist attractions included in the trip, and the AI lists famous European landmarks such as the Eiffel Tower in Paris, the Colosseum in Rome, the Brandenburg Gate in Berlin, and Rhine Falls in Zurich, highlighting that travelers will experience iconic sites in each city.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Is travel insurance included?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes, basic travel insurance is provided for all travelers. This insurance includes medical expense coverage and support in emergency situations. Enhanced coverage is available upon request.', additional_kwargs={}, response_

## vector DB Retriever Memory

In [4]:
import faiss
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores.faiss import FAISS

In [6]:
embeddings_model = OpenAIEmbeddings()

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model, index, InMemoryDocstore({}), {})

In [7]:
from langchain.memory import VectorStoreRetrieverMemory

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
memory = VectorStoreRetrieverMemory(retriever=retriever)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11932\758164859.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(retriever=retriever)


In [8]:
# Save arbitrary conversations
memory.save_context(
    inputs={
        "human": "Hello, thank you for coming to the interview today. Please introduce yourself."
    },
    outputs={
        "ai": "Hello. I'm a junior developer who majored in Computer Science. In college, I mainly used Java and Python, and recently, I participated in a web development project where I gained experience developing services for real users."
    },
)
memory.save_context(
    inputs={"human": "What was your role in the project?"},
    outputs={
        "ai": "My role was as a backend developer. I was responsible for processing user data and developing server logic, implementing RESTful APIs for communication with the frontend. I also participated in database design."
    },
)
memory.save_context(
    inputs={
        "human": "If you faced any difficulties in team projects, how did you resolve them?"
    },
    outputs={
        "ai": "We had some communication issues at the beginning of the project. To resolve this, our team held regular meetings to share each person's progress. Also, when problems arose, we actively shared opinions and worked to find reasonable solutions."
    },
)
memory.save_context(
    inputs={"human": "What do you consider your strengths as a developer?"},
    outputs={
        "ai": "My strengths are quick learning ability and problem-solving skills. I can quickly acquire new technologies and tools, and when faced with complex problems, I can propose creative solutions. Also, I value teamwork and consider collaboration with colleagues important."
    },
)

In [9]:
# Query about educational background
print("Query: What was the interviewee's major?")
print(
    memory.load_memory_variables({"prompt": "What was the interviewee's major?"})[
        "history"
    ]
)

Query: What was the interviewee's major?
human: Hello, thank you for coming to the interview today. Please introduce yourself.
ai: Hello. I'm a junior developer who majored in Computer Science. In college, I mainly used Java and Python, and recently, I participated in a web development project where I gained experience developing services for real users.


## SQLite memory

In [10]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

In [11]:
chat_message_history = SQLChatMessageHistory(session_id="sql_history", connection="sqlite:///sqlite.db")

In [12]:
# Add a user message
chat_message_history.add_user_message(
    "Hello, nice to meet you! My name is Heesun :) I'm a LangChain developer. I look forward to working with you!"
)
# Add an AI message
chat_message_history.add_ai_message(
    "Hi, Heesun! Nice to meet you. I look forward to working with you too!"
)

In [13]:
chat_message_history.messages

[HumanMessage(content="Hello, nice to meet you! My name is Heesun :) I'm a LangChain developer. I look forward to working with you!", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi, Heesun! Nice to meet you. I look forward to working with you too!', additional_kwargs={}, response_metadata={})]

In [14]:
# Clear the session memory
chat_message_history.clear()
chat_message_history.messages

[]

In [15]:
from langchain_core.messages import HumanMessage

In [16]:
# Add a user message with additional metadata.
user_message = HumanMessage(
    content="Can you help me summarize this text?",
    additional_kwargs={"task": "summarization"},
)

# Add the message to chat history.
chat_message_history.add_message(user_message)

In [17]:
chat_message_history.messages

[HumanMessage(content='Can you help me summarize this text?', additional_kwargs={'task': 'summarization'}, response_metadata={})]

In [18]:
from langchain_core.messages import AIMessage

# Add an AI message with response metadata.
ai_message = AIMessage(
    content="Sure! Here's the summary of the provided text.",
    response_metadata={"model": "gpt-4", "token_count": 30, "response_time": "150ms"},
)

In [19]:
chat_message_history.add_message(ai_message)


In [20]:
chat_message_history.messages

[HumanMessage(content='Can you help me summarize this text?', additional_kwargs={'task': 'summarization'}, response_metadata={}),
 AIMessage(content="Sure! Here's the summary of the provided text.", additional_kwargs={}, response_metadata={'model': 'gpt-4', 'token_count': 30, 'response_time': '150ms'})]

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [22]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        # Placeholder for chat history
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

# Chaining
chain = prompt | ChatOpenAI(model_name="gpt-4o") | StrOutputParser()

In [23]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db"
    )

In [24]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

In [25]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key='question',
    history_messages_key='chat_history',
    history_factory_config=config_fields
)

In [26]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [27]:
chain_with_history.invoke(
    {"question": "Hi, nice to meet you. My name is Heesun."}, config
)

'Hello Heesun! Nice to meet you too. How can I assist you today?'

In [28]:
chain_with_history.invoke({"question": "What is my name?"}, config)

'Your name is Heesun.'

In [29]:
# Config settings
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation2"}}

# Execute by passing the question and config
chain_with_history.invoke({"question": "What is my name?"}, config)

"I'm sorry, I do not have access to personal information about you. My responses are based only on the information provided in our conversation. Is there anything else I could help you with today?"